In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if not (project_root / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [3]:
from src.data.api_client import APIClientFactory
from src.data.data_validator import DataValidator

In [4]:
import pandas as pd
import numpy as np

In [5]:
client = APIClientFactory.get_primary_client()
validator = DataValidator()

In [6]:
historical_aqi = client.fetch_historical("Lahore", 31.558, 74.351, "2026-07-20", "2026-07-25")
historical_weather = client.fetch_historical_weather("Lahore", 31.558, 74.351, "2026-07-20", "2026-07-25")

2026-07-26 14:15:38 | INFO     | OpenMeteoClient:79 | Fetching historical data from https://air-quality-api.open-meteo.com/v1/air-quality for Lahore [2026-07-20 -> 2026-07-25]
2026-07-26 14:15:39 | INFO     | OpenMeteoClient:79 | Fetching historical data from https://api.open-meteo.com/v1/forecast for Lahore [2026-07-20 -> 2026-07-25]


In [7]:
df_aqi, df_weather = pd.DataFrame(historical_aqi), pd.DataFrame(historical_weather)

In [8]:
df= pd.merge(df_weather, df_aqi, on=["date", "city", "lat", "lon"])

## Features Generation

#### Temporal Features

These are usually human-readable features extracted from timestamp and information like the high AQI on weekdays or weekends will help identify the pattern.

In [9]:
# 1. Temporal features
df["hour"] = df["date"].dt.hour
df["day_of_week"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

#### Cyclic Encoding

Keep continuos and cyclic nature of timely features like looping of time and months. Example like 23 hour (11pm) is far from 0 hour but actually on clock sit right next to each other.
</br>
Trigonometric functions are used to keep such nature.  

In [10]:
# 2. Cyclical encoding 
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

In [11]:
# Group once, reused for all per-city time-series ops below
g = df.groupby("city")["european_aqi"]

#### Lag Hours

Historical values of target variable from previous time steps as it provides historical context and autocorrelation like what happens recently will predicts what will happen next

In [12]:
LAG_HOURS = [1, 3]
# 3. Lag features
for lag in LAG_HOURS:
    df[f"aqi_lag_{lag}h"] = g.shift(lag)

#### Rolling windows

Aggregated mean and standard deviation on across a sliding window of historical rows. Rolling mean smooth the noise and rolling Std tells the volatility or stability of signal. It tells the model the consistency and spikes in the aqi trend 

In [13]:
ROLLING_WINDOWS = [1, 6]
# 4. Rolling statistics (shift(1) first so window never includes current row)

for window in ROLLING_WINDOWS:
    df[f"aqi_roll_mean_{window}h"] = (
        df.groupby("city")["european_aqi"]
        .transform(lambda x: x.shift(1).rolling(window).mean())
    )
    df[f"aqi_roll_std_{window}h"] = (
        df.groupby("city")["european_aqi"]
        .transform(lambda x: x.shift(1).rolling(window).std())
    )

#### Rate of Change

Delta between previous step and current value expose the momentum and direction over short and long horizons. Example AQI reading of 80 that was 30 an hour ago tells that air become more pollutant.

In [14]:
# 5. Rate of change
df["aqi_change_1h"] = g.shift(1).diff(1) # keep it to t-1
df["aqi_change_24h"] = g.shift(1).diff(24)

#### Domain Interaction Features

Sometimes processes are driven by compound conditions so mathematical combination of raw features helps in identification of patterns that individual can not. It also helps model in learning non linear interactions. 

In [15]:
# 6. Weather interactions
df["temp_humidity"] = df["temperature_2m"] * df["relative_humidity_2m"]
df["wind_pm25"] = df["wind_speed_10m"] / (df["pm2_5"] + 1)  # +1 avoids div by zero

In [18]:
forecast_horizon = 1

In [19]:
# 7. Target creation (shift AQI forward, per city)
for day in range(1, forecast_horizon + 1):
    df[f"aqi_next_{day}d"] = df.groupby("city")["european_aqi"].shift(-day * 24)

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 38 columns):
 #   Column                 Non-Null Count  Dtype                       
---  ------                 --------------  -----                       
 0   date                   144 non-null    datetime64[ns, Asia/Karachi]
 1   city                   144 non-null    object                      
 2   lat                    144 non-null    float64                     
 3   lon                    144 non-null    float64                     
 4   temperature_2m         144 non-null    float64                     
 5   relative_humidity_2m   144 non-null    float64                     
 6   wind_speed_10m         144 non-null    float64                     
 7   rain                   144 non-null    float64                     
 8   pm10                   144 non-null    float64                     
 9   pm2_5                  144 non-null    float64                     
 10  carbon_monoxid

In [21]:
df.describe()

,lat,lon,temperature_2m,relative_humidity_2m,wind_speed_10m,rain,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,...,aqi_lag_3h,aqi_roll_mean_1h,aqi_roll_std_1h,aqi_roll_mean_6h,aqi_roll_std_6h,aqi_change_1h,aqi_change_24h,temp_humidity,wind_pm25,aqi_next_1d
count,144.000,144.000,144.000000,144.000000,144.000000,144.000000,144.000000,144.000000,144.000000,144.000000,...,141.000000,143.000000,0.0,138.000000,138.000000,142.000000,119.000000,144.000000,144.000000,120.000000
mean,31.558,74.351,28.338889,84.069919,9.364159,0.302778,60.119445,39.950695,540.277778,21.786111,...,77.004654,76.866547,NaN,76.378348,1.857421,-0.326584,-9.610449,2365.587718,0.330258,68.757301
std,0.000,0.000,2.195254,8.407520,2.729372,0.694327,63.254859,24.854891,182.040972,12.656564,...,21.966554,21.842355,NaN,21.519385,1.759986,1.955708,19.883718,105.470007,0.242101,13.322531
min,31.558,74.351,24.450001,60.360767,3.797104,0.000000,11.400000,9.300000,263.000000,3.900000,...,45.883339,45.883339,NaN,47.866671,0.100500,-7.575760,-47.934998,2043.211883,0.047762,45.883339
25%,31.558,74.351,26.850000,78.359146,7.570172,0.000000,23.275000,21.975000,422.000000,13.200000,...,63.013336,63.014999,NaN,62.964862,0.483753,-0.990831,-26.268337,2300.264355,0.145542,62.835000
50%,31.558,74.351,28.075000,84.868954,9.026953,0.000000,40.049999,33.000000,484.500000,19.500000,...,67.353333,67.353333,NaN,66.974949,1.410877,-0.196669,-0.820000,2359.232442,0.243191,64.763336
75%,31.558,74.351,29.625000,90.649708,11.304132,0.225000,61.200000,47.550000,616.000000,28.050000,...,90.603333,88.409164,NaN,84.941113,2.714558,0.309244,3.645004,2440.040158,0.470976,69.888332
max,31.558,74.351,33.849998,98.514183,16.848999,4.500000,323.200012,119.000000,1186.000000,70.400002,...,122.641678,122.641678,NaN,121.604730,7.213756,16.159084,17.526657,2622.858446,1.260193,112.118340
